In [ ]:
import meshio as mio
import igl
import numpy as np
import h5py
from collections import deque


In [ ]:
index=200

# path = "/Users/teseo/Downloads/Embryogram test/20250510_tracking-analysis-07_19T18_47-analysis.hdf5"
path = "/Users/teseo/Downloads/Embryogram test/still.hdf5"


In [ ]:
sim_path = f"outnz/sim{index}.vtu"
sim = mio.read(sim_path)

In [ ]:
T = sim.cells[0].data
V = sim.points

In [ ]:
hdf5_file = h5py.File(path, "r")
Vi, Ti = hdf5_file["mesh/v"][:].astype(float), hdf5_file["mesh/t"][:].astype(np.int32)
middle = hdf5_file["bc/middle"][:].astype(np.int32)
TT, _ = igl.tet_tet_adjacency(Ti)

In [ ]:
q = deque()
q.append(0)
ind = np.zeros(T.shape[0], dtype=np.int32)

faces = np.array([[0, 1, 2],
                  [0, 1, 3],
                  [1, 2, 3],
                  [2, 0, 3]])

iiii = 0
while q:
    t = q.popleft()
    if ind[t] == 1:
        continue

    ind[t] = 1
    for tti in range(4):
        tt = TT[t][tti]
        if tt == -1 or ind[tt] != 0:
            continue
        ftt = Ti[t, faces[tti]]
        # print(faces[TTi[tt,tti]], ftt)

        bad_face = 0
        for ftti in ftt:
            if ftti in middle:
                bad_face += 1

        if bad_face == 3:
            continue
        q.append(tt)

    iiii += 1
    if iiii % 10000 == 0:
        print(iiii, len(q), np.sum(ind), T.shape[0])



In [ ]:
np.sum(ind), T.shape

In [ ]:
sim.cell_data["ind"] = [ind]

In [ ]:
sim.write(f"outnz/sim{index}_m.vtu")